In [ ]:
from pathlib import Path

# Find repo root
REPO_ROOT = Path.cwd().parent
print(f"Repo root: {REPO_ROOT}")

REPORT_ROOT = REPO_ROOT / "report"


In [ ]:
import pandas as pd
import geopandas as gpd
from pathlib import Path
import sys
import json
from shapely.geometry import shape

# Add src to path for imports
sys.path.insert(0, str(Path.cwd().parent / 'src'))

from hotelling.spatial.boundaries import load_boundary

PATH_RAW = REPO_ROOT / Path('data/raw')

# Midpoint table (center coordinates)
zensus = gpd.read_parquet(PATH_RAW / 'zensus2022_grid.parquet')
zensus_filtered = gpd.read_parquet(PATH_RAW / 'zensus2022_grid_filtered.parquet')
lor = gpd.read_parquet(PATH_RAW / 'lor_shapes.parquet')

# CRITICAL FIX: berlin.geojson has EPSG:3035 coordinates but geopandas auto-detects as EPSG:4326
# We must force the correct CRS instead of transforming from the wrong one
with open(PATH_RAW / 'city_boundary_Berlin.geojson', 'r') as f:
    berlin_json = json.load(f)
berlin = gpd.GeoDataFrame([1], geometry=[shape(berlin_json['geometry'])], crs='EPSG:3035')

In [ ]:
boundary = load_boundary(PATH_RAW / 'relation_boundary_14983.geojson')
boundary.plot()

In [ ]:
import matplotlib.pyplot as plt

# ===== PLOTTING OPTIONS =====
plot_lor = True  # Plot LOR district boundaries
plot_boundary = True  # Plot Berlin boundary

# ===== FIGURE CONFIGURATION =====
# Note: changing these values does NOT affect the marker scaling property
fig_width_inches = 10
fig_height_inches = 10
dpi = 100

# Calculate appropriate marker size for 100x100m grid
bounds = zensus_filtered.total_bounds  # [minx, miny, maxx, maxy]
width_m = bounds[2] - bounds[0]  # width in meters
height_m = bounds[3] - bounds[1]  # height in meters

# Convert figure size to pixels
fig_width_px = fig_width_inches * dpi
fig_height_px = fig_height_inches * dpi

# Calculate pixels per meter
px_per_meter_x = fig_width_px / width_m
px_per_meter_y = fig_height_px / height_m

# 100x100m grid cell size in pixels (use smaller dimension to be conservative)
grid_cell_px = min(px_per_meter_x, px_per_meter_y) * 100

# Convert pixels to matplotlib points (1 point ≈ 1/72 inch)
# markersize is in points, so: pixels / (dpi/72) = pixels * 72 / dpi
markersize = grid_cell_px * 72 / dpi

# Create figure
fig, ax = plt.subplots(figsize=(fig_width_inches, fig_height_inches), dpi=dpi)

# Plot population grid
zensus_filtered.plot(column='Einwohner', markersize=markersize, cmap='viridis', legend=True, ax=ax, marker ='s')  # Use square markers for better grid representation

# Plot LOR boundaries if enabled
if plot_lor:
    lor.plot(ax=ax, alpha=1, edgecolor='black', facecolor='none', linewidth=1)

# Plot Berlin boundary if enabled
if plot_boundary:
    boundary.plot(ax=ax, alpha=1, edgecolor='firebrick', facecolor='none', linewidth=2)

ax.set_title('Population by 100×100m Grid Cells')
plt.show()

print(f"Map bounds: {width_m:.0f}m × {height_m:.0f}m")
print(f"Calculated marker size: {markersize:.1f} points")
print(f"LOR boundaries plotted: {plot_lor}")
print(f"Berlin boundary plotted: {plot_boundary}")


In [ ]:
import folium
from folium.raster_layers import ImageOverlay
import numpy as np
import matplotlib.colors as mcolors
import rasterio
from rasterio.features import geometry_mask
from rasterio.transform import Affine
from rasterio.warp import reproject, Resampling, calculate_default_transform
from pyproj import CRS, Transformer
import tempfile
from PIL import Image
from shapely.geometry import box, shape
import json

# ===== INTERACTIVE MAP OPTIONS =====
plot_lor_interactive = True
plot_boundary_interactive = True
grid_opacity = 0.75

# Work in original EPSG:3035 projection (meters)
zensus_3035 = zensus_filtered.copy()
berlin_3035 = berlin.copy()

# CRITICAL FIX: Reconstruct 100m x 100m squares from midpoint coordinates
# The x_mp_100m and y_mp_100m are the midpoints of 100m grid cells
# Create square polygons by offsetting ±50m from the midpoint
half_cell = 50  # 50 meters = half of 100m cell

grid_squares = []
for idx, row in zensus_3035.iterrows():
    x_mid = row['x_mp_100m']
    y_mid = row['y_mp_100m']
    
    # Create 100m x 100m square centered on the midpoint
    square = box(x_mid - half_cell, y_mid - half_cell, x_mid + half_cell, y_mid + half_cell)
    grid_squares.append(square)

# Replace point geometry with square geometry
zensus_3035_polygons = zensus_3035.copy()
zensus_3035_polygons['geometry'] = grid_squares

# Clip 100m squares to the Berlin boundary so edge cells don't
# visually protrude beyond the boundary line.
berlin_union = berlin_3035.geometry.unary_union
zensus_3035_polygons['geometry'] = zensus_3035_polygons.geometry.intersection(berlin_union)
# Drop any degenerate geometries that collapsed to points/lines after clipping
zensus_3035_polygons = zensus_3035_polygons[
    ~zensus_3035_polygons.geometry.is_empty
    & (zensus_3035_polygons.geometry.geom_type.isin(['Polygon', 'MultiPolygon']))
].copy()
print(f"Cells after boundary clip: {len(zensus_3035_polygons)}")

# Fix CRS for boundary and lor files - they should already be in EPSG:3035 but ensure consistency
lor_3035 = lor.copy()
if lor_3035.crs is None or str(lor_3035.crs) != 'EPSG:3035':
    if str(lor_3035.crs) == 'EPSG:4326':
        lor_3035 = lor_3035.set_crs('EPSG:3035', allow_override=True)
    else:
        lor_3035 = lor_3035.to_crs('EPSG:3035')

boundary_3035 = load_boundary(PATH_RAW / 'relation_boundary_14983.geojson')
# load_boundary already returns EPSG:3035, but ensure it's correct
if boundary_3035.crs is None or str(boundary_3035.crs) != 'EPSG:3035':
    boundary_3035 = boundary_3035.set_crs('EPSG:3035', allow_override=True)

berlin_3035 = berlin.copy()
# berlin is already correctly EPSG:3035 from Cell 2, just ensure consistency
if berlin_3035.crs is None or str(berlin_3035.crs) != 'EPSG:3035':
    berlin_3035 = berlin_3035.set_crs('EPSG:3035', allow_override=True)

# Get WGS84 versions for Folium display
zensus_wgs84 = zensus_3035_polygons.to_crs('EPSG:4326')
lor_wgs84 = lor_3035.to_crs('EPSG:4326')
boundary_wgs84 = boundary_3035.to_crs('EPSG:4326')
berlin_wgs84 = berlin_3035.to_crs('EPSG:4326')

# Calculate map center (in WGS84)
center_lat = zensus_wgs84.geometry.centroid.y.mean()
center_lon = zensus_wgs84.geometry.centroid.x.mean()

# Create base map
m = folium.Map(
    location=[center_lat, center_lon],
    zoom_start=12,
    tiles='OpenStreetMap'
)

# Rasterize in EPSG:3035 (meters) with proper 100m resolution
print("Creating raster from 100m x 100m grid squares...")
min_pop = zensus_3035_polygons['Einwohner'].min()
max_pop = zensus_3035_polygons['Einwohner'].max()

# Get bounds in EPSG:3035
bounds_3035 = zensus_3035_polygons.total_bounds  # [minx, miny, maxx, maxy]
minx_proj, miny_proj, maxx_proj, maxy_proj = bounds_3035

# CRITICAL FIX: Adjust bounds to account for grid cell edges
# The bounds above are from centroid coordinates (with .mod 100 = 50)
# Grid cells extend ±50m from centroids, so actual grid edges are:
minx_proj -= 50  # Grid cell left edge
miny_proj -= 50  # Grid cell bottom edge
maxx_proj += 50  # Grid cell right edge
maxy_proj += 50  # Grid cell top edge

# Raster resolution: 100m per pixel (one pixel = one grid cell)
resolution = 100  # meters per pixel
width = int(np.ceil((maxx_proj - minx_proj) / resolution))
height = int(np.ceil((maxy_proj - miny_proj) / resolution))

print(f"Raster size: {width}x{height} pixels (100m/pixel = {width*100}m x {height*100}m)")

# Create raster array (RGBA)
raster = np.zeros((height, width, 4), dtype=np.uint8)

# Create affine transform for EPSG:3035
transform = Affine.translation(minx_proj, maxy_proj) * Affine.scale(resolution, -resolution)

# Rasterize each grid cell
print(f"Rasterizing {len(zensus_3035_polygons)} grid cells...")
for idx, row in zensus_3035_polygons.iterrows():
    geom = row.geometry
    pop = row['Einwohner']
    
    # Normalize population to color
    if max_pop == min_pop:
        norm_value = 0.5
    else:
        norm_value = (pop - min_pop) / (max_pop - min_pop)
    norm_value = np.clip(norm_value, 0, 1)
    
    # Get RGBA color from viridis
    color_rgba = plt.cm.viridis(norm_value)
    color_uint8 = (np.array(color_rgba) * 255).astype(np.uint8)
    
    # Create mask for this geometry
    try:
        mask = geometry_mask(
            [geom],
            out_shape=(height, width),
            transform=transform,
            invert=True  # True where geometry is
        )
        # Apply color to masked pixels
        raster[mask] = color_uint8
    except Exception as e:
        pass

print("Raster created successfully")

# Reproject raster from EPSG:3035 to EPSG:3857 (Web Mercator)
src_crs = CRS.from_epsg(3035)
dst_crs = CRS.from_epsg(3857)

# Source raster shape and transform (already computed above)
src_height, src_width = raster.shape[:2]

# Compute the optimal output transform + dimensions in EPSG:3857
dst_transform, dst_width, dst_height = calculate_default_transform(
    src_crs,
    dst_crs,
    src_width,
    src_height,
    left=minx_proj,
    bottom=miny_proj,
    right=maxx_proj,
    top=maxy_proj,
)

# Allocate destination array (same RGBA dtype)
raster_3857 = np.zeros((dst_height, dst_width, 4), dtype=np.uint8)

# Reproject each of the 4 channels independently
for band_idx in range(4):
    src_band = raster[:, :, band_idx].astype(np.float32)
    dst_band = np.zeros((dst_height, dst_width), dtype=np.float32)
    reproject(
        source=src_band,
        destination=dst_band,
        src_transform=transform,
        src_crs=src_crs,
        dst_transform=dst_transform,
        dst_crs=dst_crs,
        resampling=Resampling.nearest,
    )
    raster_3857[:, :, band_idx] = np.clip(dst_band, 0, 255).astype(np.uint8)

print(f"Reprojected raster size: {dst_width}x{dst_height} pixels (EPSG:3857)")

# Derive WGS84 bounds from the EPSG:3857 extent
minx_3857 = dst_transform.c
maxy_3857 = dst_transform.f
maxx_3857 = minx_3857 + dst_transform.a * dst_width
miny_3857 = maxy_3857 + dst_transform.e * dst_height

# Convert EPSG:3857 corners to WGS84 lat/lon
t = Transformer.from_crs(dst_crs, CRS.from_epsg(4326), always_xy=True)
minx_wgs84, miny_wgs84 = t.transform(minx_3857, miny_3857)
maxx_wgs84, maxy_wgs84 = t.transform(maxx_3857, maxy_3857)

print(
    f"Bounds in WGS84: lat [{miny_wgs84:.4f}, {maxy_wgs84:.4f}], "
    f"lon [{minx_wgs84:.4f}, {maxx_wgs84:.4f}]"
)

# Save reprojected raster to temp file
raster_img = Image.fromarray(raster_3857, 'RGBA')
temp_file = tempfile.NamedTemporaryFile(suffix='.png', delete=False)
raster_img.save(temp_file.name)
temp_file.close()

# Overlay the raster on the map (Folium expects [[lat_min, lon_min], [lat_max, lon_max]])
ImageOverlay(
    image=temp_file.name,
    bounds=[[miny_wgs84, minx_wgs84], [maxy_wgs84, maxx_wgs84]],
    opacity=grid_opacity,
    name='Population Grid (100m x 100m)'
).add_to(m)

# Add LOR boundaries as lines
if plot_lor_interactive:
    folium.GeoJson(
        lor_wgs84.to_json(),
        style_function=lambda x: {'color': 'black', 'weight': 2, 'opacity': 0.6, 'fill': False},
        name='LOR Districts',
        show=True
    ).add_to(m)

# Add Berlin boundary as blue line (inner ringbahn)
if plot_boundary_interactive:
    folium.GeoJson(
        boundary_wgs84.to_json(),
        style_function=lambda x: {'color': 'blue', 'weight': 3, 'opacity': 1.0, 'fill': False},
        name='Ringbahn (14983)',
        show=True
    ).add_to(m)

# Add city boundary as red line
folium.GeoJson(
    berlin_wgs84.to_json(),
    style_function=lambda x: {'color': 'red', 'weight': 2, 'opacity': 0.8, 'fill': False},
    name='Berlin Boundary',
    show=True
).add_to(m)

# Add layer control
folium.LayerControl().add_to(m)

# Save map
output_path = Path(REPORT_ROOT / 'grid_map_interactive.html')
output_path.parent.mkdir(parents=True, exist_ok=True)
m.save(output_path)

print(f"\nInteractive map saved to {output_path}")
print(f"Population range: {min_pop:.0f} - {max_pop:.0f}")
print(f"Grid resolution: 100m x 100m (reconstructed from midpoints)")
print(f"Grid opacity: {grid_opacity}")
